# Suite PYDEF — Python definitions

The names a Python module binds, in the scopes Python gives them (`Python.Definitions.define`): one entity per name
per scope, with every node that binds it, and lookup outward through enclosing functions to the module, past class
bodies. Each case checks the entities, their declarations, and what names resolve to.

In [ ]:
from pathlib import Path

from mbse.Programs.Framework import Syntax as F
from mbse.Programs.Python import Definitions as D, Python314, Syntax as S
from mbse.Programs.Python._Standard import PythonStandard

PY315 = PythonStandard(3, 15)
SOURCES = Path("../../conformance/sources")


def define(text):
    module = PY315.parse(text)
    return module, D.define(module)


def listing(program):
    return [(e.kind, e.qualified_name(), len(e.declarations)) for e in program.entities()]


def names(module, spelling):
    """The `Name` nodes of `module` spelled `spelling`, in source order."""
    return [n for n in F.walk(module) if isinstance(n, S.Name) and n.id.spelling == spelling]


def refers(program, node):
    return [e.qualified_name() for e in D.referents(program, node)]

## PYDEF-01 · Names bound in a scope are local to it, one entity per name, whatever binds it

In [ ]:
module, program = define("""
x = 1
x += 2
y: int
a, (b, *c) = [d] = e = f()
for i in range(3):
    del i
with open() as (g, h.attribute), open() as k[0]:
    pass
try:
    pass
except E as err:
    pass
import os.path, sys as system
from . import mod as alias, other
from m import *
def function(): pass
class Klass: pass
""")
assert listing(program) == [
    ("variable", "x", 2), ("variable", "y", 1), ("variable", "a", 1), ("variable", "b", 1), ("variable", "c", 1),
    ("variable", "d", 1), ("variable", "e", 1), ("variable", "i", 2), ("variable", "g", 1), ("variable", "err", 1),
    ("import", "os", 1), ("import", "system", 1), ("import", "alias", 1), ("import", "other", 1),
    ("function", "function", 1), ("class", "Klass", 1)]
x = program.lookup("x")[0]
assert x.definition is names(module, "x")[0] and x.declarations == names(module, "x")
assert program.entity_of(names(module, "x")[1]) is x and program.root.kind == "module" and program.root.separator == "."
assert program.lookup("function")[0].definition is module.body[-2] and program.lookup("Klass")[0].scope.kind == "class"
assert program.scope_of(module) is program.root and program.lookup("nothing") == []
assert refers(program, names(module, "f")[0]) == [] and refers(program, names(module, "h")[0]) == []  # not bound here
print(len(listing(program)), "entities")

## PYDEF-02 · Lookup goes outward through functions to the module, past class bodies; `global` and `nonlocal`

In [ ]:
module, program = define("""
counter = 0
nonlocal stray
class Outer:
    size = 1
    def method(self):
        return size, counter
    class Inner:
        value = size
    squares = [size for _ in range(size)]
def outer():
    global counter, created
    counter = 1
    created = 2
    def inner():
        nonlocal later
        later = 3
        def innermost():
            nonlocal later
            return later
    def unbound():
        nonlocal missing
        missing = 4
    later = 0
def reader():
    global ghost
    return ghost
""")
assert refers(program, names(module, "size")[1]) == []  # a method does not see its class's names
assert refers(program, names(module, "counter")[1]) == ["counter"]
assert refers(program, names(module, "size")[2]) == []  # nor does a nested class
assert refers(program, names(module, "size")[3]) == [] and refers(program, names(module, "size")[4]) == ["Outer.size"]
assert [e.qualified_name() for e in program.entities() if e.name in ("counter", "created", "later", "missing")] == [
    "counter", "created", "outer.later", "outer.unbound.missing"]
later = program.lookup("outer.later")[0]
assert len(program.lookup("counter")[0].declarations) == 2 and later.declarations == [names(module, "later")[0],
                                                                                     names(module, "later")[2]]
assert later.definition is names(module, "later")[0]  # in source order: `later = 3` in inner, then `later = 0`
assert refers(program, names(module, "later")[1]) == ["outer.later"]
ghost = program.lookup("ghost")[0]
assert ghost.declarations == [] and ghost.definition is None and refers(program, names(module, "ghost")[0]) == ["ghost"]
assert refers(program, module.body[3].body[0].names[0]) == ["counter"]  # the names of `global` itself
assert program.lookup("stray")[0].parent is program.root  # `nonlocal` outside a function finds none: its own scope
assert program.lookup("Outer.Inner.value") and program.lookup("Outer.method")[0].scope.parent is program.root
print("ok")

## PYDEF-03 · Lambdas, comprehensions and assignment expressions

In [ ]:
module, program = define("""
def f(items):
    squares = [y := x * x for x in items if (last := x)]
    nested = {k: v for k, row in items for v in row}
    total = sum(n for n in items)
    pick = lambda item, key=items: (found := item)
    return y, last, x
x = 0
""")
assert listing(program)[:4] == [("function", "f", 1), ("parameter", "f.items", 1), ("variable", "x", 1),
                                ("variable", "x", 1)]  # the module's names first, then the function's
scopes = [program.scope_of(c) for c in F.walk(module) if isinstance(c, S.Comprehension)]
assert [s.kind for s in scopes] == ["comprehension", "comprehension", "comprehension", "comprehension"]
assert scopes[0].parent is program.lookup("f")[0].scope and scopes[1] is scopes[2]
assert refers(program, names(module, "y")[1]) == ["f.y"] and refers(program, names(module, "last")[1]) == ["f.last"]
assert refers(program, names(module, "x")[-2]) == ["x"]  # the comprehension's own `x` stays in it
assert refers(program, names(module, "items")[1]) == ["f.items"]  # the first iterable is evaluated outside
assert [e.kind for e in program.entities() if e.name in ("item", "key", "found")] == ["parameter", "parameter",
                                                                                      "variable"]
lam = next(n for n in F.walk(module) if isinstance(n, S.Lambda))
assert program.scope_of(lam.args.args[1].default_value) is program.lookup("f")[0].scope  # defaults outside
assert program.scope_of(lam.body).kind == "lambda" and refers(program, names(module, "found")[0]) == ["found"]
print("ok")

## PYDEF-04 · Parameters, decorators, defaults, annotations, and type parameters' annotation scopes

In [ ]:
module, program = define("""
T = int
@decorator(T)
def generic[T: bound, *Ts, **P = default](a: T, /, b: T = T, *c: *Ts, d, **e: P) -> T:
    return a
class Box[T](Base[T], metaclass=Meta):
    item: T
type Alias[K] = dict[K, T]
type Plain = T
""")
function = program.lookup("generic")[0]
parameters = [e for e in program.entities() if e.parent is function.scope]
assert [(e.kind, e.name) for e in parameters] == [("parameter", n) for n in "abcde"]
annotations = function.scope.parent
assert annotations.kind == "type parameters" and [e.name for e in annotations.entities()] == ["T", "Ts", "P"]
assert annotations.parent is program.root and annotations.node is module.body[1]
uses = [[e.kind for e in D.referents(program, use)] for use in names(module, "T")]
assert uses == [["variable"], ["variable"],  # `T = int`, and the decorator's: the module's
                ["type parameter"], ["type parameter"], ["variable"],  # the annotations' are the parameter, a default the module's
                ["type parameter"], ["type parameter"], ["type parameter"],  # the return annotation's, a base's, and in the class body
                ["variable"], ["variable"]]  # the type aliases', whose own parameters are others
box = program.lookup("Box")[0]
assert box.scope.parent.kind == "type parameters"
assert refers(program, names(module, "Meta")[0]) == []
alias = program.lookup("Alias")[0]
assert alias.kind == "type alias" and alias.definition is module.body[3] and program.entity_of(module.body[3].name) is alias
print("ok")

## PYDEF-05 · Pattern captures, and names a pattern only reads

In [ ]:
module, program = define("""
match command:
    case [first, *rest] | {"k": first, **rest}:
        pass
    case Point(x=0, y=py) as point if point:
        pass
    case constants.RED | _:
        pass
    case {"k": key}:
        pass
""")
assert listing(program) == [("variable", "first", 2), ("variable", "rest", 2), ("variable", "py", 1),
                            ("variable", "point", 1), ("variable", "key", 1)]
assert refers(program, names(module, "point")[0]) == ["point"] and refers(program, names(module, "constants")[0]) == []
klass = next(n for n in F.walk(module) if isinstance(n, S.MatchClass))
assert D.referents(program, klass.kwd_attrs[0]) == []  # a keyword names an attribute
print("ok")

## PYDEF-06 · What names refer to: attributes, keywords and modules' names find nothing, and lookup takes dotted names

In [ ]:
module, program = define("""
import a.b
class C:
    def m(self):
        return self.m(key=1)
""")
attribute = next(n for n in F.walk(module) if isinstance(n, S.Attribute))
keyword = next(n for n in F.walk(module) if isinstance(n, S.Keyword))
assert D.referents(program, attribute.attr) == [] and D.referents(program, keyword.arg) == []
assert refers(program, attribute.value) == ["C.m.self"] and refers(program, attribute.value.id) == ["C.m.self"]
dotted = module.body[0].names[0].name
assert refers(program, dotted.names[0]) == ["a"] and D.referents(program, dotted.names[1]) == []
assert program.lookup("C.m.self")[0].kind == "parameter" and program.lookup("C.x") == []
assert program.lookup("self", at=attribute) == program.lookup("C.m.self") and repr(program.lookup("C.m")[0]) == "<function C.m>"
assert D.referents(program, S.Identifier(spelling="C")) == []  # not in the program
print("ok")

## PYDEF-07 · Every entity of the conformance source, and every name located

In [ ]:
module = Python314.parse((SOURCES / "python314.py").read_text(encoding="utf-8"))
program = D.define(module)
entities = listing(program)
assert entities[:6] == [("import", "annotations", 1), ("import", "osp", 1), ("import", "sys", 1),
                        ("import", "sibling", 1), ("import", "one", 1), ("import", "second", 1)]
assert ("function", "Shape.__init__", 1) in entities and ("variable", "statements.counter", 3) in entities
assert ("type alias", "Pair", 1) in entities and ("variable", "patterns.origin", 1) in entities
assert all(program.scope_of(n) is not None for n in F.walk(module) if isinstance(n, S.Name))
print(len(entities), "entities")